# Combining DataFrames

Real analysis almost always joins multiple tables — like the rows of a restaurant database:

- **`concat`** — stack tables on top of each other (more rows).
- **`merge`** — combine tables side by side based on a shared key (like SQL `JOIN`).
- **`join`** — a convenience merge on the index.

This notebook uses the same data a restaurant point-of-sale system would store.

In [1]:
import pandas as pd

In [2]:
# Local execution already has the data files.
# On Colab (which opens just this notebook) we download them:
import os
import urllib.request

NEEDS_FILES = ['Restaurant - Week 1 Sales.csv', 'Restaurant - Week 2 Sales.csv', 'Restaurant - Customers.csv', 'Restaurant - Foods.csv', 'Restaurant - Week 1 Satisfaction.csv']

for _f in NEEDS_FILES:
    if not os.path.exists(_f):
        urllib.request.urlretrieve('https://raw.githubusercontent.com/sivabalanb/Data-Analysis-with-Pandas-and-Python/master/' + _f, _f)
        print('downloaded', _f)


## The restaurant data

- `week1` / `week2` — the same shape: sales records.
- `customers` — a lookup table of customer details.
- `foods` — a lookup table of menu items.

In [3]:
week1 = pd.read_csv("Restaurant - Week 1 Sales.csv")
week2 = pd.read_csv("Restaurant - Week 2 Sales.csv")
customers = pd.read_csv("Restaurant - Customers.csv")
foods = pd.read_csv("Restaurant - Foods.csv")

In [4]:
len(week1)

250

In [5]:
len(week2)

250

## `concat` — stacking rows

`pd.concat([week1, week2])` stacks one below the other. With `ignore_index=True` the row labels are renumbered instead of repeating. The `keys=` argument adds a MultiIndex level that tags which table each row came from.

In [6]:
len(pd.concat([week1,week2]))

500

In [7]:
pd.concat([week1,week2],ignore_index=True)

,Customer ID,Food ID
0,537,9
1,97,4
2,658,1
3,202,2
4,155,9
...,...,...
495,783,10
496,556,10
497,547,9
498,252,9


In [8]:
pd.concat([week1,week2], keys=["Week1","Week2"])

Customer ID  Food ID
Week1 0            537        9
      1             97        4
      2            658        1
      3            202        2
      4            155        9
...                ...      ...
Week2 245          783       10
      246          556       10
      247          547        9
      248          252        9
      249          249        6

[500 rows x 2 columns]

In [9]:
week1 = pd.read_csv("Restaurant - Week 1 Sales.csv")
week2 = pd.read_csv("Restaurant - Week 2 Sales.csv")
customers = pd.read_csv("Restaurant - Customers.csv")
foods = pd.read_csv("Restaurant - Foods.csv")

In [10]:
pd.concat([week1, week2], ignore_index=True).tail()

,Customer ID,Food ID
495,783,10
496,556,10
497,547,9
498,252,9
499,249,6


## `merge` — joining on a key

`merge` behaves like a SQL `INNER JOIN` by default: only rows whose keys match in *both* tables survive.

- `on=` names the shared column.
- `suffixes=` disambiguates columns that exist in both tables.
- Joining on multiple keys (`["Customer ID", "Food ID"]`) matches only rows that agree on *all* of them.

In [11]:
week1.merge(week2)

,Customer ID,Food ID
0,304,3
1,540,3
2,937,10
3,233,3
4,21,4
5,922,1
6,21,4
7,578,5
8,578,5


In [12]:
week1.merge(week2, how='inner', on='Customer ID',suffixes=[" - week1"," - week2"])

,Customer ID,Food ID - week1,Food ID - week2
0,537,9,5
1,155,9,3
2,503,5,8
3,503,5,9
4,155,1,3
...,...,...,...
57,945,5,4
58,343,3,5
59,343,3,2
60,343,3,7


In [13]:
week1.merge(week2, how='inner', on=['Customer ID','Food ID'],suffixes=[" - week1"," - week2"])

,Customer ID,Food ID
0,304,3
1,540,3
2,937,10
3,233,3
4,21,4
5,922,1
6,21,4
7,578,5
8,578,5


In [14]:
week1.merge(week2, how='inner', suffixes=[" - week1"," - week2"])

,Customer ID,Food ID
0,304,3
1,540,3
2,937,10
3,233,3
4,21,4
5,922,1
6,21,4
7,578,5
8,578,5


In [15]:
week1[week1['Customer ID']==21]

,Customer ID,Food ID
101,21,4
212,21,4


In [16]:
week2[week2['Customer ID']==21]

,Customer ID,Food ID
30,21,4


## `outer` merges and the `indicator`

`how="outer"` keeps every row from both tables, filling gaps with `NaN`. `indicator=True` adds a `_merge` column that records where each row came from — `both_only`, `left_only`, or `right_only`. This is how you audit a join.

In [17]:
outer_merge = week1.merge(week2, how='outer',on="Customer ID", suffixes=[" - week1"," - week2"], indicator=True)

In [18]:
outer_merge.nunique()

Customer ID        399
Food ID - week1     10
Food ID - week2     10
_merge               3
dtype: int64

In [19]:
outer_merge["_merge"].value_counts()

_merge
right_only    197
left_only     195
both           62
Name: count, dtype: int64

## `left` join — enrich one table

`week1.merge(foods, how="left", on="Food ID")` keeps every `week1` row and adds the matching food details, leaving `NaN` where no food matched.

In [20]:
foods.head()

,Food ID,Food Item,Price
0,1,Sushi,3.99
1,2,Burrito,9.99
2,3,Taco,2.99
3,4,Quesadilla,4.25
4,5,Pizza,2.49


In [21]:
week1.merge(foods, how='left', on='Food ID' )

,Customer ID,Food ID,Food Item,Price
0,537,9,Donut,0.99
1,97,4,Quesadilla,4.25
2,658,1,Sushi,3.99
3,202,2,Burrito,9.99
4,155,9,Donut,0.99
...,...,...,...,...
245,413,9,Donut,0.99
246,926,6,Pasta,13.99
247,134,3,Taco,2.99
248,396,6,Pasta,13.99


In [22]:
week1.merge(foods, how='left', on='Food ID' , sort=True)

,Customer ID,Food ID,Food Item,Price
0,658,1,Sushi,3.99
1,600,1,Sushi,3.99
2,155,1,Sushi,3.99
3,341,1,Sushi,3.99
4,20,1,Sushi,3.99
...,...,...,...,...
245,809,10,Drink,1.75
246,584,10,Drink,1.75
247,274,10,Drink,1.75
248,151,10,Drink,1.75


In [23]:
week2.head()

,Customer ID,Food ID
0,688,10
1,813,7
2,495,10
3,189,5
4,267,3


## Different key names

When the keys have different names, use `left_on=` and `right_on=`. The `.drop("ID", axis="columns")` removes the redundant key column.

In [24]:
customers.head()

,ID,First Name,Last Name,Gender,Company,Occupation
0,1,Joseph,Perkins,Male,Dynazzy,Community Outreach Specialist
1,2,Jennifer,Alvarez,Female,DabZ,Senior Quality Engineer
2,3,Roger,Black,Male,Tagfeed,Account Executive
3,4,Steven,Evans,Male,Fatz,Registered Nurse
4,5,Judy,Morrison,Female,Demivee,Legal Assistant


In [25]:
week2.merge(customers, left_on='Customer ID', right_on='ID', how = 'left').drop('ID',axis='columns')

,Customer ID,Food ID,First Name,Last Name,Gender,Company,Occupation
0,688,10,Carl,Williamson,Male,Thoughtmix,Graphic Designer
1,813,7,Johnny,Walker,Male,Kayveo,Developer II
2,495,10,Deborah,Little,Female,Babbleblab,VP Accounting
3,189,5,Roger,Gordon,Male,Skilith,Operator
4,267,3,Matthew,Wood,Male,Agimba,Product Engineer
...,...,...,...,...,...,...,...
245,783,10,Phyllis,Meyer,Female,Voolia,Information Systems Manager
246,556,10,Samuel,Bailey,Male,Oyoloo,Nurse
247,547,9,Tina,Watkins,Female,Thoughtstorm,Accountant II
248,252,9,Douglas,Powell,Male,Jetwire,Geologist IV


## `join` — merge on the index

`week1.join(satisfaction)` merges on the *index* instead of a column. Use it when both tables are already keyed by their index.

In [26]:
satifaction = pd.read_csv("Restaurant - Week 1 Satisfaction.csv")

In [27]:
satifaction.head()

,Satisfaction Rating
0,2
1,7
2,3
3,7
4,10


In [28]:
week1.join(satifaction)

,Customer ID,Food ID,Satisfaction Rating
0,537,9,2
1,97,4,7
2,658,1,3
3,202,2,7
4,155,9,10
...,...,...,...
245,413,9,1
246,926,6,2
247,134,3,8
248,396,6,10
